# Unit 2: Indexing Mastery — .loc, .iloc, Boolean Masks 🟡

**Estimated Time:** 2 hours (condensed for speed)  
**Prerequisites:** Unit 1 complete  
**You already know:** How Pandas aligns on Index labels

---

## Why This Matters in Quant Finance 🎯

**The Scenario:**

You're analyzing 1000 stocks over 10 years. You need to:
- Find days where tech stocks moved >3% while energy dropped
- Extract Q4 returns for correlation analysis
- Flag outlier moves (±3 standard deviations) for risk review
- Backtest a strategy that only trades on high-volume days

**With poor indexing skills:** 30 minutes of trial-and-error, StackOverflow, and bugs  
**With mastery:** 30 seconds, one line of code

Row/column selection is **80% of data work**. Master `.loc`, `.iloc`, and boolean masks → 10x productivity.

---

## Learning Objectives 📚

1. **Choose correctly** between `.loc[]` (label-based) and `.iloc[]` (position-based)
2. **Write boolean masks** to filter data on complex conditions
3. **Avoid SettingWithCopyWarning** and understand when you're working on a view vs a copy
4. **Chain operations safely** without breaking indexing
5. **Build filters** for outlier detection and conditional selection

---

## Core Concepts — Condensed 🔍

### 1. `.loc[]` — Label-Based Selection

**Syntax:** `df.loc[row_labels, column_labels]`

```python
# Single row, all columns
returns_df.loc['2024-01-15']

# Single row, single column
returns_df.loc['2024-01-15', 'AAPL']

# Slice of rows, specific columns
returns_df.loc['2024-01-01':'2024-01-31', ['AAPL', 'TSLA']]

# All rows, one column (returns a Series)
returns_df.loc[:, 'AAPL']
# Equivalent to: returns_df['AAPL']
```

**Key:** Uses **Index labels** (dates, tickers), not positions.

---

### 2. `.iloc[]` — Position-Based Selection

**Syntax:** `df.iloc[row_positions, column_positions]`

```python
# First row (position 0), all columns
returns_df.iloc[0]

# First row, first column
returns_df.iloc[0, 0]

# First 10 rows, first 2 columns
returns_df.iloc[:10, :2]

# Last row
returns_df.iloc[-1]
```

**Key:** Uses **integer positions** (0, 1, 2...), like NumPy arrays. Ignores labels entirely.

---

### 3. When to Use Which? 🎯

| **Use `.loc[]` when:** | **Use `.iloc[]` when:** |
|------------------------|-------------------------|
| Working with dates/tickers | Working with positions (first, last, nth) |
| Slicing by time periods | Selecting by row number |
| Using boolean masks | Iterating with numeric index |
| Default choice for finance | Rare in quant work |

**Rule of thumb:** In finance, use `.loc[]` 95% of the time.

---

### 4. Boolean Masks — The Power Tool 🔥

**A boolean mask is a Series of True/False matching your DataFrame's index.**

```python
# Create a mask
mask = returns_df['AAPL'] > 0.02  # True where AAPL > 2%

# Apply the mask
big_days = returns_df[mask]  # Only rows where mask is True

# One-liner (most common)
big_days = returns_df[returns_df['AAPL'] > 0.02]
```

**Combining conditions:**

```python
# AND: Both conditions must be True
mask = (returns_df['AAPL'] > 0.02) & (returns_df['TSLA'] < -0.01)

# OR: Either condition can be True
mask = (returns_df['AAPL'] > 0.02) | (returns_df['TSLA'] > 0.02)

# NOT: Invert the condition
mask = ~(returns_df['AAPL'] > 0.02)  # AAPL <= 2%
```

**CRITICAL:** Use `&` (and), `|` (or), `~` (not) — NOT Python's `and`, `or`, `not`!

---

### 5. SettingWithCopyWarning — The Infamous Gotcha ⚠️

**The warning:**
```
SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame
```

**What it means:** You're modifying what might be a view (reference to original) or a copy. Pandas isn't sure, so it warns you.

**Common trigger:**
```python
# BAD: Chained indexing
df[df['AAPL'] > 0]['new_col'] = 1  # ❌ Might not work!

# GOOD: Single .loc[] operation
df.loc[df['AAPL'] > 0, 'new_col'] = 1  # ✅ Always works
```

**The fix:** Use `.loc[]` for all assignment operations. Never chain indexing when setting values.

---

## Hands-On Tasks 💻

### Task 1: Master `.loc[]` and `.iloc[]` (30 min)

```python
# Use your returns_df from Unit 1

# TODO 1.1: Extract returns for the first week of January 2024
# Use .loc[] with date slice

# TODO 1.2: Get the first 5 rows using .iloc[]

# TODO 1.3: Get AAPL returns for March 15-20 using .loc[]
# Hint: returns_df.loc['2024-03-15':'2024-03-20', 'AAPL']

# TODO 1.4: Get the last row, last column using .iloc[]

# TODO 1.5: Compare performance
# Time .loc[] vs .iloc[] for selecting 100 random rows
# Which is faster? Why?

# TODO 1.6: What happens if you mix them?
# Try: returns_df.loc[0, 'AAPL']  # Using position 0 with .loc
# Try: returns_df.iloc['2024-01-02', 0]  # Using date with .iloc
# Document the errors you get
```

---

### Task 2: Boolean Masking for Trading Signals (45 min)

```python
# TODO 2.1: Find all days where AAPL gained more than 2%
# Create mask, apply it, print first 10 results

# TODO 2.2: Find days where AAPL AND TSLA both gained >1%
# Use & operator
# Q: How many such days exist?

# TODO 2.3: Find days where ANY stock dropped more than 3%
# Hint: (col1 < -0.03) | (col2 < -0.03) | (col3 < -0.03)
# Or use: (returns_df < -0.03).any(axis=1)

# TODO 2.4: Find days where all stocks moved in the same direction
# All positive OR all negative
# Hint: (returns_df > 0).all(axis=1) | (returns_df < 0).all(axis=1)

# TODO 2.5: Filter to Q4 2024 using .loc and boolean mask
# Two methods:
# Method A: returns_df.loc['2024-10-01':'2024-12-31']
# Method B: mask = (returns_df.index >= '2024-10-01') & (returns_df.index <= '2024-12-31')
# Which is cleaner?
```

---

### Task 3: Avoiding SettingWithCopyWarning (30 min)

```python
# TODO 3.1: Trigger the warning (on purpose)
# Create a filtered DataFrame
big_moves = returns_df[returns_df['AAPL'] > 0.02]
# Try to add a column
big_moves['flag'] = 'extreme'  # May trigger warning

# TODO 3.2: Fix it with .loc[]
# Use .loc[] to add the column to the original DataFrame
returns_df.loc[returns_df['AAPL'] > 0.02, 'flag'] = 'extreme'

# TODO 3.3: Explicit copy when you want a separate DataFrame
big_moves_copy = returns_df[returns_df['AAPL'] > 0.02].copy()
big_moves_copy['flag'] = 'extreme'  # No warning, separate object

# TODO 3.4: Understand view vs copy
# Which creates a view? Which creates a copy?
a = returns_df['AAPL']  # View or copy?
b = returns_df[['AAPL', 'TSLA']]  # View or copy?
c = returns_df[returns_df > 0]  # View or copy?
# Test by modifying and seeing if original changes
```

---

### Task 4: Build an Outlier Detector (15 min)

```python
# TODO 4.1: Compute statistics
mean_returns = returns_df.mean()
std_returns = returns_df.std()

# TODO 4.2: Define outliers (>2 std devs from mean)
# For each stock, mark returns that are extreme
# Hint: Use broadcasting
outlier_threshold_high = mean_returns + 2 * std_returns
outlier_threshold_low = mean_returns - 2 * std_returns

# TODO 4.3: Create boolean mask for outliers
# True where ANY stock has an outlier
is_outlier = (returns_df > outlier_threshold_high) | (returns_df < outlier_threshold_low)
has_outlier = is_outlier.any(axis=1)  # True if any column is an outlier

# TODO 4.4: Extract outlier days
outlier_days = returns_df[has_outlier]
print(f"Found {len(outlier_days)} days with outlier moves")
print(outlier_days.head(10))
```

---

## Self-Check Questions 🧠

**Q1:** When do you use `.loc[]` vs `.iloc[]`? Give an example of each.

**Q2:** What's wrong with this code?
```python
df[df['returns'] > 0.02]['ticker'] = 'high'
```

**Q3:** How do you select rows where AAPL > 2% AND TSLA < -1%? Write the code.

**Q4:** Why does boolean indexing preserve the Index while `.iloc[]` can change it?

**Q5:** Fix this code to avoid SettingWithCopyWarning:
```python
high_returns = df[df['AAPL'] > 0.05]
high_returns['category'] = 'extreme'
```

---

## End of Unit Deliverable 🎯

**Build:** Function that identifies "extreme move" days and returns a clean report.

```python
def identify_extreme_moves(returns_df, n_std=2):
    """
    Identify trading days with extreme moves (>n standard deviations).
    
    Parameters:
    -----------
    returns_df : pd.DataFrame
        DataFrame with stock returns, DatetimeIndex
    n_std : float, default=2
        Number of standard deviations to define "extreme"
    
    Returns:
    --------
    report_df : pd.DataFrame
        Columns: date, stock, return, z_score, direction
        Only rows with extreme moves
        Sorted by absolute z-score (most extreme first)
    
    Example:
    --------
    >>> report = identify_extreme_moves(returns_df, n_std=2)
    >>> print(report.head())
           date      stock    return  z_score direction
    0  2024-03-15   TSLA    0.0523     2.34      up
    1  2024-07-22   AAPL   -0.0412    -2.15    down
    ...
    """
    # TODO: Implement this
    pass
```

**Specs:**
- Compute mean and std for each stock
- Calculate z-scores: `(return - mean) / std`
- Flag returns where `|z-score| > n_std`
- Return long-format DataFrame (one row per extreme event)
- Include: date, stock ticker, actual return, z-score, direction ('up' or 'down')
- Sort by absolute z-score descending

**Test your function:**
```python
extreme_report = identify_extreme_moves(returns_df, n_std=2)
print(f"Total extreme events: {len(extreme_report)}")
print(extreme_report.head(10))
```

---

## Exit Criteria ✅

You're ready for Unit 3 when you can:

1. [ ] Write `.loc[]` queries for date ranges without checking docs
2. [ ] Combine 3+ boolean conditions with `&`, `|`, `~`
3. [ ] Explain why chained indexing causes SettingWithCopyWarning
4. [ ] Filter a DataFrame to specific conditions in one line
5. [ ] Debug indexing errors by checking if you need `.loc[]` or `.iloc[]`

---

**Ready to start?** Let me know when you finish each task and I'll spot-check your work before you move to the deliverable! 🚀

**Estimated pace:** 
- Tasks 1-2: 1 hour
- Tasks 3-4: 30 min
- Deliverable: 30 min
- **Total: ~2 hours**